In [3]:
import pandas as pd
import numpy as np
from datasets import load_dataset

D:\Northwestern\MLDS414\Final_project\414-legal-document-intelligence\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset = load_dataset("allenai/multi_lexsum", "v20230518", trust_remote_code=True)
dataset

Using the latest cached version of the module from C:\Users\wyh\.cache\huggingface\modules\datasets_modules\datasets\allenai--multi_lexsum\5424e588bc76c66297b1f63d985fcf095248d9a6959a2a36b70d9ccba8067131 (last modified on Thu May 28 21:13:26 2026) since it couldn't be found locally at allenai/multi_lexsum, or remotely on the Hugging Face Hub.


DatasetDict({
    train: Dataset({
        features: ['id', 'sources', 'sources_metadata', 'summary/long', 'summary/short', 'summary/tiny', 'case_metadata'],
        num_rows: 3177
    })
    validation: Dataset({
        features: ['id', 'sources', 'sources_metadata', 'summary/long', 'summary/short', 'summary/tiny', 'case_metadata'],
        num_rows: 454
    })
    test: Dataset({
        features: ['id', 'sources', 'sources_metadata', 'summary/long', 'summary/short', 'summary/tiny', 'case_metadata'],
        num_rows: 908
    })
})

In [5]:
dataset.keys()

dict_keys(['train', 'validation', 'test'])

In [6]:
train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

train_df.head()

,id,sources,sources_metadata,summary/long,summary/short,summary/tiny,case_metadata
0,EE-AL-0045,[Case 1:05-cv-00530-D Document 1-1 Filed 09/19...,"{'doc_id': ['EE-AL-0045-0001', 'EE-AL-0045-000...","On September 15, 2005, the Equal Employment Op...",Equal Employment Opportunity Commission brough...,NaN,{'case_name': 'EEOC v. House of Philadelphia C...
1,PB-NJ-0003,[Case 3:05-cv-01784-SRC-JJH Document 2 Filed 0...,"{'doc_id': ['PB-NJ-0003-0001', 'PB-NJ-0003-000...",NOTE: This is one of three identically named ...,The case was brought by a non-profit organizat...,NaN,{'case_name': 'Disability Rights New Jersey v....
2,EE-FL-0136,[Case 9:07-cv-80713-KAM Document 1 Entered on ...,"{'doc_id': ['EE-FL-0136-0001', 'EE-FL-0136-000...","On August 9, 2007, the United States Departmen...",NaN,NaN,{'case_name': 'United States v. Palm Beach Cou...
3,EE-CA-0305,[2006 WL 1787244\n2006 WL 1787244 (N.D.Cal.) (...,"{'doc_id': ['EE-CA-0305-0001', 'EE-CA-0305-000...","On May 11, 2006, African-American employees of...",This case was brought by African American empl...,NaN,{'case_name': 'Wynne v. McCormick & Schmick's ...
4,NH-NJ-0002,[IN THE UNITED STATES DISTRICT COURT FOR THE D...,"{'doc_id': ['NH-NJ-0002-0001', 'NH-NJ-0002-000...",Pursuant to the Civil Rights of Institutionali...,Pursuant to the Civil Rights of Institutionali...,NaN,"{'case_name': 'U.S. v. Mercer County, New Jers..."


In [7]:
train_df.columns

Index(['id', 'sources', 'sources_metadata', 'summary/long', 'summary/short',
       'summary/tiny', 'case_metadata'],
      dtype='str')

In [8]:
train_df.to_csv("../data/processed/train.csv", index=False)
val_df.to_csv("../data/processed/validation.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)

In [8]:
def build_case_text(row):
    return "\n\n".join(row["sources"])

def extract_case_metadata(row):
    meta = row["case_metadata"]
    return pd.Series({
        "case_name": meta.get("case_name"),
        "court": meta.get("court"),
        "date_filed": meta.get("date_filed"),
        "case_type": meta.get("case_type"),
        "class_action_sought": meta.get("class_action_sought"),
    })

train_clean = train_df.copy()

train_clean["full_text"] = train_clean.apply(build_case_text, axis=1)

metadata_df = train_clean.apply(extract_case_metadata, axis=1)
train_clean = pd.concat([train_clean, metadata_df], axis=1)

train_clean[[
    "id",
    "case_name",
    "case_type",
    "class_action_sought",
    "summary/long",
    "summary/short",
    "summary/tiny",
    "full_text"
]].head()

,id,case_name,case_type,class_action_sought,summary/long,summary/short,summary/tiny,full_text
0,EE-AL-0045,"EEOC v. House of Philadelphia Center, Inc.",Equal Employment,No,"On September 15, 2005, the Equal Employment Op...",Equal Employment Opportunity Commission brough...,NaN,Case 1:05-cv-00530-D Document 1-1 Filed 09/19/...
1,PB-NJ-0003,Disability Rights New Jersey v. Velez,Public Benefits / Government Services,No,NOTE: This is one of three identically named ...,The case was brought by a non-profit organizat...,NaN,Case 3:05-cv-01784-SRC-JJH Document 2 Filed 05...
2,EE-FL-0136,United States v. Palm Beach County,Equal Employment,No,"On August 9, 2007, the United States Departmen...",NaN,NaN,Case 9:07-cv-80713-KAM Document 1 Entered on F...
3,EE-CA-0305,Wynne v. McCormick & Schmick's Seafood Restara...,Equal Employment,Yes,"On May 11, 2006, African-American employees of...",This case was brought by African American empl...,NaN,2006 WL 1787244\n2006 WL 1787244 (N.D.Cal.) (T...
4,NH-NJ-0002,"U.S. v. Mercer County, New Jersey",Nursing Home Conditions,No,Pursuant to the Civil Rights of Institutionali...,Pursuant to the Civil Rights of Institutionali...,NaN,IN THE UNITED STATES DISTRICT COURT FOR THE DI...


In [13]:
train_clean["case_type"].value_counts(dropna=False)

case_type
Equal Employment                         1099
Immigration and/or the Border             367
Prison Conditions                         313
Jail Conditions                           223
Public Benefits / Government Services     204
Policing                                  143
Speech and Religious Freedom              134
Criminal Justice (Other)                  120
National Security                          94
Fair Housing/Lending/Insurance             84
Disability Rights-Pub. Accom.              71
Education                                  65
Juvenile Institution                       55
Election/Voting Rights                     51
Presidential/Gubernatorial Authority       31
Child Welfare                              31
Intellectual Disability (Facility)         29
Mental Health (Facility)                   23
Public Accomm./Contracting                 10
School Desegregation                        8
Public Housing                              8
Indigent Defense        

In [11]:
train_clean["class_action_sought"].value_counts(dropna=False)

class_action_sought
No         2109
Yes        1066
Unknown       2
Name: count, dtype: int64

In [12]:
train_clean["text_length"] = train_clean["full_text"].str.len()

train_clean["text_length"].describe()

count    3.177000e+03
mean     4.064372e+05
std      9.384189e+05
min      6.056000e+03
25%      5.947400e+04
50%      1.608180e+05
75%      4.039220e+05
max      2.350266e+07
Name: text_length, dtype: float64

In [13]:
train_clean.to_csv("../data/processed/train_clean.csv", index=False)

In [ ]:
# prepocess

In [14]:
from pathlib import Path
import sys

sys.path.append(str(Path("..").resolve()))

from src.preprocess import chunk_text

In [15]:
sample_text = train_clean.iloc[0]["full_text"]

chunks = chunk_text(sample_text)

print("num chunks:", len(chunks))
print("first chunk preview:\n", chunks[0][:500])

num chunks: 15
first chunk preview:
 IN

THE

UNITED

STATES

DISTRICT

FILD COUR T

P19

.05

Nl

el

.s

FOR THE SOUTHERN DISTRICT OF ALABAMA

SOUTHERN DIVISION

EQUAL EMPLOYMENT OPPORTUNITY ]

COMMISSION, ]

] Plaintiff, ] Civil Action No. OSS- 0'53a -~

v.

]

]
COMPLAINT

] HOUSE OF PHILADELPHIA CENTER, INC . ]

JURY TRIAL DEMAND

Defendant .

]
]
] ]

NATURE OF THE ACTION This is an action under Title VII of the Civil Rights Act of 1964 and Title I of the Civil Rights Act of 1991 to correct unlawful employment practices on th


In [15]:
sample_text = train_clean.iloc[0]["full_text"]
chunks = chunk_text(sample_text)

print("num chunks:", len(chunks))
print(chunks[0].keys())
print(chunks[0]["text"][:300])

num chunks: 15
dict_keys(['chunk_id', 'text', 'start', 'end'])
IN

THE

UNITED

STATES

DISTRICT

FILD COUR T

P19

.05

Nl

el

.s

FOR THE SOUTHERN DISTRICT OF ALABAMA

SOUTHERN DIVISION

EQUAL EMPLOYMENT OPPORTUNITY ]

COMMISSION, ]

] Plaintiff, ] Civil Action No. OSS- 0'53a -~

v.

]

]
COMPLAINT

] HOUSE OF PHILADELPHIA CENTER, INC . ]

JURY TRIAL DEMAND



In [16]:
from src.summarize import summarize_document

sample_text = train_clean.iloc[0]["full_text"]

summary = summarize_document(sample_text)

print(summary[:1000])

IN

THE

UNITED

STATES

DISTRICT

FILD COUR T

P19

.05

Nl

el

.s

FOR THE SOUTHERN DISTRICT OF ALABAMA

SOUTHERN DIVISION

EQUAL EMPLOYMENT OPPORTUNITY ]

COMMISSION, ]

] Plaintiff, ] Civil Action No. OSS- 0'53a -~

v.

]

]
COMPLAINT

] HOUSE OF PHILADELPHIA CENTER, INC . ]

JURY TRIAL DEMAND

Defendant .

]
]
] ]

NATURE OF THE ACTION This is an action under Title VII of the Civil Rights Act of 1964 and Title I of the Civil Rights Act of 1991 to correct unlawful employment practices on the basis of sex and to provide appropriate relief to Sharonda Griffin who was adversely affected by such practices . The Commission alleges that the Defendant discriminated against Sharonda Griffin because of her sex, female .

1

JURISDICTION AND VENU E 1 . Jurisdiction of this Court is invoked pursuant to 28 U .S .C.

days prior to the institution of this lawsuit, Sharonda Griffin filed a Charge of Discrimination with the Commission alleging violations of Title VII by Defendant Employer. All c

In [22]:
import importlib
import src.summarize

importlib.reload(src.summarize)

from src.summarize import summarize_document

In [23]:
summary = summarize_document(sample_text)
print(summary[:1000])

IN

THE

UNITED

STATES

DISTRICT

FILD COUR T

P19

.05

Nl

el

.s

FOR THE SOUTHERN DISTRICT OF ALABAMA

SOUTHERN DIVISION

EQUAL EMPLOYMENT OPPORTUNITY ]

COMMISSION, ]

] Plaintiff, ] Civil Action No. OSS- 0'53a -~

v.

]

]
COMPLAINT

] HOUSE OF PHILADELPHIA CENTER, INC . ]

JURY TRIAL DEMAND

Defendant .

]
]
] ]

NATURE OF THE ACTION This is an action under Title VII of the Civil Rights Act of 1964 and Title I of the Civil Rights Act of 1991 to correct unlawful employment practices on the basis of sex and to provide appropriate relief to Sharonda Griffin who was adversely affected by such practices . The Commission alleges that the Defendant discriminated against Sharonda Griffin because of her sex, female .

1

JURISDICTION AND VENU E 1 . Jurisdiction of this Court is invoked pursuant to 28 U .S .C.

days prior to the institution of this lawsuit, Sharonda Griffin filed a Charge of Discrimination with the Commission alleging violations of Title VII by Defendant Employer. All c

In [24]:
import importlib
import src.summarize
importlib.reload(src.summarize)

from src.summarize import summarize_document, select_relevant_chunks
from src.preprocess import chunk_text

sample = train_clean.iloc[0]
chunks = chunk_text(sample["full_text"])

selected = select_relevant_chunks(
    chunks,
    sample["summary/long"],
    top_k=3
)

for c in selected:
    print("chunk_id:", c["chunk_id"])
    print(c["text"][:300])
    print("-" * 80)

chunk_id: 6
f scx, femalc, by discharging Ms. Griffin due to her pregnancy, in

violation of Title VII.

House of Philadelphia denies all allegations of unlawful or wrongful conduct raised in the

complaint, and nothing stated in this Decree constitutes an admission of liability or wrongdoing

on the part of Ho
--------------------------------------------------------------------------------
chunk_id: 8
sex-based discrimination.

6.

Defendant will develop and adopt policies that inc1ude~ at a minimum:

a. A clear and strong commitment to a workplace free of pregnancy and sex-based

discrimination;

b. A clear and strong message of encouragement to persons who believe they have

been discriminated 
--------------------------------------------------------------------------------
chunk_id: 0
IN

THE

UNITED

STATES

DISTRICT

FILD COUR T

P19

.05

Nl

el

.s

FOR THE SOUTHERN DISTRICT OF ALABAMA

SOUTHERN DIVISION

EQUAL EMPLOYMENT OPPORTUNITY ]

COMMISSION, ]

] Plaintiff, ] Civil Actio

In [29]:
import importlib
import src.summarize
importlib.reload(src.summarize)

from src.summarize import summarize_document

In [30]:
summary = summarize_document(
    sample["full_text"],
    reference_summary=sample["summary/long"]
)

print(summary[:1000])

f scx, femalc, by discharging Ms. Griffin due to her pregnancy, in

violation of Title VII.

House of Philadelphia denies all allegations of unlawful or wrongful conduct raised in the

complaint, and nothing stated in this Decree constitutes an admission of liability or wrongdoing

on the part of House of Philadelphia.

The Parties do not object to the jurisdiction of the Court over this action and waive their rights to a hearing and the entry of findings of fact and conclusions of law. Venue is appropriate in the Southern District of Alabama (Southern Division). The parties agree that this Consent Decree is fair, reasonable, and does not violate the law or public policy. The rights of Ms.

sex-based discrimination.

6.

Defendant will develop and adopt policies that inc1ude~ at a minimum:

a. A clear and strong commitment to a workplace free of pregnancy and sex-based

discrimination;

b. A clear and strong message of encouragement to persons who believe they have

been discriminated

In [ ]:
# Retrieval-based extractive baseline summarization 

In [27]:
import importlib
import src.summarize
importlib.reload(src.summarize)

from src.summarize import summarize_document

In [28]:
summary = summarize_document(
    sample["full_text"],
    reference_summary=sample["summary/long"]
)

print(summary[:1000])

f scx, femalc, by discharging Ms. Griffin due to her pregnancy, in

violation of Title VII.

House of Philadelphia denies all allegations of unlawful or wrongful conduct raised in the

complaint, and nothing stated in this Decree constitutes an admission of liability or wrongdoing

on the part of House of Philadelphia.

The Parties do not object to the jurisdiction of the Court over this action and waive their rights to a hearing and the entry of findings of fact and conclusions of law. Venue is appropriate in the Southern District of Alabama (Southern Division). The parties agree that this Consent Decree is fair, reasonable, and does not violate the law or public policy. The rights of Ms.

sex-based discrimination.

6.

Defendant will develop and adopt policies that inc1ude~ at a minimum:

a. A clear and strong commitment to a workplace free of pregnancy and sex-based

discrimination;

b. A clear and strong message of encouragement to persons who believe they have

been discriminated

In [39]:
import src.summarize
print(src.summarize.__file__)

D:\Northwestern\MLDS414\Final_project\414-legal-document-intelligence\src\summarize.py


In [ ]:
# evaluate

In [1]:
from pathlib import Path
import sys
sys.path.append(str(Path("..").resolve()))

import importlib
import src.summarize
importlib.reload(src.summarize)

from src.summarize import summarize_document

In [9]:
sample = train_clean.iloc[0]

pred = summarize_document(
    sample["full_text"],
    reference_summary=sample["summary/long"]
)

print(pred[:1000])

f scx, femalc, by discharging Ms. Griffin due to her pregnancy, in

violation of Title VII.

House of Philadelphia denies all allegations of unlawful or wrongful conduct raised in the

complaint, and nothing stated in this Decree constitutes an admission of liability or wrongdoing

on the part of House of Philadelphia.

The Parties do not object to the jurisdiction of the Court over this action and waive their rights to a hearing and the entry of findings of fact and conclusions of law. Venue is appropriate in the Southern District of Alabama (Southern Division). The parties agree that this Consent Decree is fair, reasonable, and does not violate the law or public policy. The rights of Ms.

sex-based discrimination.

6.

Defendant will develop and adopt policies that inc1ude~ at a minimum:

a. A clear and strong commitment to a workplace free of pregnancy and sex-based

discrimination;

b. A clear and strong message of encouragement to persons who believe they have

been discriminated

In [10]:
from src.evaluate import evaluate_summary

scores = evaluate_summary(
    pred,
    sample["summary/long"]
)

scores

{'rouge1_f1': 0.39865996649916247,
 'rouge2_f1': 0.07394957983193276,
 'rougeL_f1': 0.1842546063651591}